<a href="https://colab.research.google.com/github/mantrikaran/F1.Stats.Guy/blob/main/Driver_Wins_and_Podiums.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

!pip install duckdb --quiet
print("✅ DuckDB installed")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ DuckDB installed


In [9]:
import duckdb

CSV_PATH = '/content/drive/MyDrive/F1 Stats Guy - Jolpica/Jolpica Database/'

conn = duckdb.connect()

views = ['rounds', 'race_results', 'drivers', 'teams']

for v in views:
    conn.execute(f"""
        CREATE OR REPLACE VIEW {v} AS
        SELECT * FROM read_csv_auto('{CSV_PATH}{v}.csv')
    """)
    count = conn.execute(f"SELECT COUNT(*) FROM {v}").fetchone()[0]
    print(f"✅ {v}: {count:,} rows")

✅ rounds: 1,173 rows
✅ race_results: 25,961 rows
✅ drivers: 818 rows
✅ teams: 205 rows


In [15]:
import re

SQL_FOLDER = '/content/drive/MyDrive/F1 Stats Guy - Jolpica/Codes/SQL Codes'

files = [
    'agent1a_driver_wins_milestones.sql',
    'agent1a_driver_podiums_milestones.sql',
]

for filename in files:
    path = f"{SQL_FOLDER}/{filename}"
    with open(path, 'r') as f:
        sql = f.read()

    if 'tr.circuit_name' in sql:
        print(f"⏭️  {filename}: already patched")
        continue

    # More flexible regex — handles varying whitespace
    patched = re.sub(
        r'(tr\.round_name\s+AS\s+trigger_race_name)',
        r'tr.round_name        AS trigger_race_name,\n        tr.circuit_name,\n        tr.country_code',
        sql
    )

    if 'tr.circuit_name' not in patched:
        print(f"❌ {filename}: patch failed — printing first match context for debug")
        # Show what the pattern actually looks like in the file
        sample = re.search(r'.{50}trigger_race_name.{50}', sql)
        print(sample.group() if sample else "Pattern not found at all")
        continue

    with open(path, 'w') as f:
        f.write(patched)

    count = sql.count('AS trigger_race_name')
    print(f"✅ {filename}: patched {count} CTEs")


⏭️  agent1a_driver_wins_milestones.sql: already patched
⏭️  agent1a_driver_podiums_milestones.sql: already patched


In [16]:
from google.colab import auth
from google.auth import default
import gspread

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_ID = '1wlZBishj6NAs4wU_qNaWKfURw03IY_gXMUipxWhWVXg'
SHEET_GID      = 1623554071

sh        = gc.open_by_key(SPREADSHEET_ID)
worksheet = next(ws for ws in sh.worksheets() if ws.id == SHEET_GID)
print(f"✅ Connected to: '{worksheet.title}'")


✅ Connected to: 'Stat Table'


In [17]:
latest = conn.execute("""
    SELECT rnd.round_id, rnd.round_name, rnd.year, rnd.round_number
    FROM   rounds rnd
    WHERE  EXISTS (
        SELECT 1 FROM race_results rr WHERE rr.round_id = rnd.round_id
    )
    ORDER  BY rnd.year DESC, rnd.round_number DESC
    LIMIT  1
""").fetchone()

trigger_round_id = latest[0]
print(f"✅ Latest race : {latest[1]} {latest[2]} (Round {latest[3]})")
print(f"   round_id    : {trigger_round_id}")


✅ Latest race : Miami Grand Prix 2026 (Round 4)
   round_id    : round_01d4Qacj


In [19]:
import pandas as pd
from datetime import datetime

assert trigger_round_id, "Run Cell 5 first"

SQL_FOLDER     = '/content/drive/MyDrive/F1 Stats Guy - Jolpica/Codes/SQL Codes'
DATE_GENERATED = datetime.now().strftime('%Y-%m-%d')

# ── Get circuit_name + country_code from rounds table ────────
race_meta = conn.execute("""
    SELECT circuit_name, country_code
    FROM   rounds
    WHERE  round_id = ?
""", [trigger_round_id]).fetchone()

circuit_name = race_meta[0]
country_code = race_meta[1]
print(f"✅ Race meta: {circuit_name} | {country_code}")

# ── Run both SQL files ────────────────────────────────────────
def run_sql(filename, metric_label):
    with open(f"{SQL_FOLDER}/{filename}", 'r') as f:
        sql = f.read()
    sql = sql.replace('$trigger_round_id', f"'{trigger_round_id}'")
    df = conn.execute(sql).df()
    df['_metric'] = metric_label
    return df

df_wins    = run_sql('agent1a_driver_wins_milestones.sql',    'Win')
df_podiums = run_sql('agent1a_driver_podiums_milestones.sql', 'Podium')
df_all     = pd.concat([df_wins, df_podiums], ignore_index=True)

print(f"  Wins    : {len(df_wins)} milestone(s)")
print(f"  Podiums : {len(df_podiums)} milestone(s)")
print(f"  Total   : {len(df_all)} rows to write")

# ── Map to sheet columns ──────────────────────────────────────
rows = []
for _, row in df_all.iterrows():
    rows.append([
        row['milestone_id'],       # Milestone ID
        DATE_GENERATED,            # Date Generated
        int(row['season']),        # Race Year
        row['milestone_label'],    # Stat Description (Milestone)
        row['driver_name'],        # Driver Name
        row['team_name'],          # Constructor Name
        row['trigger_race_name'],  # Race Name
        circuit_name,              # Race Track
        country_code,              # Track Country
        row['_metric'],            # Metric
        'Post Race',               # Dimension
    ])

worksheet.append_rows(rows, value_input_option='USER_ENTERED')
print(f"\n✅ {len(rows)} rows written to '{worksheet.title}'")


✅ Race meta: Miami International Autodrome | USA
  Wins    : 9 milestone(s)
  Podiums : 59 milestone(s)
  Total   : 68 rows to write

✅ 68 rows written to 'Stat Table'
